# Delta Lake — Partitioning vs. Liquid Clustering

## Objetivos del ejercicio

En este notebook vamos a trabajar con dos estrategias de organización física de datos en Delta Lake:

| Estrategia | Descripción |
|---|---|
| **Partitioning** | Divide los datos en subdirectorios físicos según el valor de una columna |
| **Liquid Clustering** | Organiza los datos dentro de los archivos Parquet por rangos de una columna, sin subdirectorios |

### Flujo del ejercicio
1. Generar un dataset de 5 millones de filas
2. Guardar como tabla Delta **particionada por `year`** → explorar subdirectorios en S3
3. Intentar aplicar Liquid Clustering sobre la tabla particionada → **error esperado**
4. Crear una nueva tabla con **Liquid Clustering by `id`** vía CTAS
5. Ejecutar `OPTIMIZE` y verificar que los archivos están ordenados por `id`
6. Añadir nuevos datos y observar el comportamiento post-append

## 0. Configuración de locations

Definimos el usuario actual y construimos las rutas S3 donde se guardarán las tablas.  
Esto permite que cada alumno trabaje en su propio subdirectorio sin colisiones.

In [0]:
import re

# Get current user from the active Databricks session
user = spark.sql("SELECT current_user()").first()[0]

# Clean username: remove domain, replace special chars with underscores
user_clean = re.sub(r"@.*", "", user)
user_clean = user_clean.replace(".", "_").replace("-", "_")

# Base S3 path for this user
base_path = f"s3://mi-bucket-publico-javier-2026/tables/{user_clean}"

# Individual table locations
partitioned_location = f"{base_path}/partitioned_employees"
clustered_location   = f"{base_path}/clustered_employees"

print(f"User:                 {user_clean}")
print(f"Partitioned location: {partitioned_location}")
print(f"Clustered location:   {clustered_location}")

User:                 test_data_jm
Partitioned location: s3://mi-bucket-publico-javier-2026/tables/test_data_jm/partitioned_employees
Clustered location:   s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees


## 1. Generación de datos

Creamos un DataFrame sintético con **5 millones de filas** y tres columnas:

- `id` — identificador único (0 a 4.999.999)
- `year` — año entre 2000 y 2023 (24 valores distintos → 24 particiones)
- `salary` — salario aleatorio entre 0 y 100.000

Utilizamos `spark.range()` que genera datos distribuidos de forma nativa en el cluster, sin necesidad de cargar nada desde disco.

In [0]:
from pyspark.sql.functions import col

df = spark.range(0, 5_000_000).selectExpr(
    "id",
    "2000 + (id % 24) as year",              # Cycles through years 2000-2023
    "cast(rand() * 100000 as int) as salary"  # Random salary 0-100000
)

# Cache to avoid recomputation in subsequent cells
#df.cache()

print(f"Total rows: {df.count():,}")
df.show(5)

Total rows: 5,000,000
+---+----+------+
| id|year|salary|
+---+----+------+
|  0|2000|   920|
|  1|2001| 59779|
|  2|2002| 15060|
|  3|2003| 54349|
|  4|2004| 75451|
+---+----+------+
only showing top 5 rows


## 2. Guardar tabla Delta particionada por `year`

Escribimos la tabla usando `.partitionBy("year")`.  

Esto creará **24 subdirectorios** en S3 con la estructura:
```
partitioned_employees/
├── year=2000/
│   └── part-00000-....parquet
├── year=2001/
│   └── part-00000-....parquet
└── ...
```

> 💡 Esta es la forma "clásica" de organizar datos en Data Lakes. Funciona bien cuando las queries filtran exactamente por la columna de partición. En cualquier otro caso, Spark debe leer **todas** las particiones (full scan).

In [0]:
(df.write
   .format("delta")
   .mode("overwrite")
   .partitionBy("year")
   .option("path", partitioned_location)
   .saveAsTable("partitioned_employees")
)

print("Table saved.")
print(f"Location: {partitioned_location}")
print("Expected: 24 subdirectories (year=2000 ... year=2023)")

Table saved.
Location: s3://mi-bucket-publico-javier-2026/tables/test_data_jm/partitioned_employees
Expected: 24 subdirectories (year=2000 ... year=2023)


### 2.1 Inspeccionar la tabla y sus subdirectorios

In [0]:
%sql
-- Check table metadata: location, number of files, partition columns, size
DESCRIBE DETAIL partitioned_employees;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,8386cec3-fabb-4d93-afa5-4d6490103519,workspace.default.partitioned_employees,null,s3://mi-bucket-publico-javier-2026/tables/test_data_jm/partitioned_employees,2026-04-13T15:58:18.079Z,2026-04-13T15:58:32.000Z,List(year),List(),24,39548842,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Show partition layout as Delta sees it
SHOW PARTITIONS partitioned_employees;

year
2012
2016
2014
2004
2000
2015
2007
2021
2017
2022


## 3. Intentar Liquid Clustering sobre tabla particionada

Liquid Clustering y Partitioning son **incompatibles**.  
Una tabla no puede tener ambos mecanismos activos al mismo tiempo.

La siguiente instrucción fallará con un error similar a:
```
AnalysisException: Clustering is not supported on a partitioned table.
```

> 🎯 Este error es **intencionado**. El objetivo es que quede claro que para usar Liquid Clustering hay que crear una tabla nueva sin `PARTITIONED BY`.

In [0]:
# %sql
# -- ⚠️ Expected to FAIL — Liquid Clustering cannot be applied to a partitioned table
# ALTER TABLE partitioned_employees
# CLUSTER BY (id);

## 4. Crear tabla con Liquid Clustering vía CTAS

La solución es crear una **nueva tabla** usando `CREATE TABLE AS SELECT` (CTAS), especificando `CLUSTER BY` en la definición y **omitiendo `PARTITIONED BY`**.

Además configuramos `delta.targetFileSize = 4MB` para que el `OPTIMIZE` posterior genere varios archivos (con el límite por defecto de 128MB todo cabría en 1 solo archivo).

> 💡 `CLUSTER BY` no ordena los datos en el momento de la escritura.  
> Solo **registra la intención** de clustering en los metadatos de la tabla.  
> El orden físico se materializa cuando se ejecuta `OPTIMIZE`.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE clustered_employees
    USING DELTA
    CLUSTER BY (id)
    LOCATION '{clustered_location}'
    TBLPROPERTIES (
        'delta.targetFileSize'        = '4194304',
        'delta.enableRowTracking'     = 'false',
        'delta.enableDeletionVectors' = 'false'
    )
    AS SELECT * FROM partitioned_employees
""")

print("Clustered table created.")
print(f"Location: {clustered_location}")

Clustered table created.
Location: s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees


### 4.1 Verificar que el clustering está registrado (antes del OPTIMIZE)

`DESCRIBE DETAIL` mostrará `clusteringColumns = [["id"]]` confirmando que la tabla está configurada para Liquid Clustering.  
Sin embargo, los datos **aún no están físicamente ordenados** hasta que ejecutemos `OPTIMIZE`.

In [0]:
%sql
DESCRIBE DETAIL clustered_employees;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,65837f20-af59-4423-a865-fd8c1e369bcc,workspace.default.clustered_employees,null,s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,2026-04-13T15:58:54.629Z,2026-04-13T16:07:19.000Z,List(),List(id),21,92783615,"Map(delta.enableDeletionVectors -> false, delta.enableRowTracking -> false, delta.checkpointPolicy -> v2, delta.targetFileSize -> 4194304)",3,7,"List(appendOnly, clustering, domainMetadata, invariants, v2Checkpoint)",Map(),false


## 5. OPTIMIZE — Materializar el Liquid Clustering

`OPTIMIZE` es el comando que:
1. Lee todos los archivos existentes
2. Los reescribe ordenados por la columna de clustering (`id`)
3. Respeta el `targetFileSize` configurado → generará varios archivos

Después del `OPTIMIZE` podremos verificar que:
- Hay **más de un archivo** Parquet
- Cada archivo tiene un rango de `id` **no solapado** con los demás

In [0]:
%sql
OPTIMIZE clustered_employees;

path,metrics
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(8, 1, List(3784812, 4557484, 4290770.25, 8, 34326162), List(31523214, 31523214, 3.1523214E7, 1, 31523214), 0, null, null, 0, 1, 1, 0, false, 0, 0, 1776095969806, 1776095994568, 8, 1, null, List(0, 0), null, 3, 3, 21900, 0, List(31523214, true, false, false, 0.8745919689222286, List(0.8745919689222286), 1.0, null, 0, 1, 31523214, 31523214, 0, 0, 0, null, log, 4194304, 4194304, 1, 0, 0, List(0), 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 31523214, 31523214, List(284, 1401, 0, 584, 387, 6655), 2, 1, 5, default, false, 0, null), null)"
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 8, 0, false, 0, 0, 1776095995028, 1776096001125, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, List(34326162, false, false, false, 0.8745919689222286, List(0.8745919689222286), 1.0, null, 0, 0, 0, 0, 0, 0, 0, null, log, 4194304, 4194304, 1, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(298, 6, 2667, 0, 0, 0), 2, 2, 5, default, false, 0, null), null)"
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 8, 0, true, 0, 0, 1776096001303, 1776096004405, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, List(34326162, false, false, false, 0.8745919689222286, List(0.8745919689222286), 1.0, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 2097152, 4194304, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 1641, 0, 0, 0), 15, 1, 1, null, false, 0, null), null)"


### 5.1 Verificar archivos y rangos de `id`

Inspeccionamos el `_delta_log` para ver cuántos archivos se han creado y cuál es el rango de `id` dentro de cada uno.  
Si el clustering funciona correctamente, los rangos serán **contiguos y sin solapamiento**.

In [0]:
%sql
-- Number of files and clustering columns after OPTIMIZE
DESCRIBE DETAIL clustered_employees;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,65837f20-af59-4423-a865-fd8c1e369bcc,workspace.default.clustered_employees,null,s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,2026-04-13T15:58:54.629Z,2026-04-13T15:59:53.000Z,List(),List(id),8,34326162,"Map(delta.enableDeletionVectors -> false, delta.enableRowTracking -> false, delta.checkpointPolicy -> v2, delta.targetFileSize -> 4194304)",3,7,"List(appendOnly, clustering, domainMetadata, invariants, v2Checkpoint)",Map(),false


In [0]:
import json
from pyspark.sql.functions import col, input_file_name

# Get the latest version number from the Delta log
last_version = spark.sql("DESCRIBE HISTORY clustered_employees LIMIT 1").first()["version"]

# Build the path for only that commit's json file
log_file = f"{clustered_location}/_delta_log/{str(last_version).zfill(20)}.json"

log_df = (
    spark.read.json(log_file)
         .filter(col("add").isNotNull())
         .selectExpr(
             "add.path as file",
             "get_json_object(add.stats, '$.minValues.id') as id_min",
             "get_json_object(add.stats, '$.maxValues.id') as id_max",
             "add.size as size_bytes"
         )
         .orderBy(col("id_min").cast("long"))
)

display(log_df)

file,id_min,id_max,size_bytes
part-00007-a4f915f2-23ba-42a4-b4f3-9c1552e0fe80.c000.snappy.parquet,0,635852,4358560
part-00003-7c750793-2920-4cb4-b655-a9784a4427c9.c000.snappy.parquet,635853,1245994,4198099
part-00006-3444d6d6-a224-4960-b87e-2b9f783ef94d.c000.snappy.parquet,1245995,1909599,4531366
part-00002-6ac766ff-148c-4230-9e72-d0eb33c1e326.c000.snappy.parquet,1909600,2577334,4557484
part-00001-676e0416-e087-4e01-85f0-06d2c95f192a.c000.snappy.parquet,2577335,3241136,4532492
part-00000-1c65658b-f728-484d-99de-fcf35cda1d93.c000.snappy.parquet,3241137,3850740,4194793
part-00005-f2f7186e-ff3f-4f41-843d-801e543d000d.c000.snappy.parquet,3850741,4394614,3784812
part-00004-9e48bb4b-22a1-40bb-a56c-3980096141a4.c000.snappy.parquet,4394615,4999999,4168556


## 6. Inserciones en rangos solapados — comportamiento del OPTIMIZE incremental

En esta sección realizamos **3 rondas de inserciones** usando rangos de `id` que ya existen en la tabla.

El objetivo es responder a una pregunta clave:

> *¿Qué ocurre cuando insertas datos en un rango de IDs que ya estaba clusterizado?*

Para cada ronda seguimos el mismo patrón:

1. **Append** de nuevos datos en un rango solapado
2. **Ver archivos activos ANTES del OPTIMIZE** → solapamiento visible
3. **Ejecutar OPTIMIZE** → re-clustering incremental
4. **Ver archivos activos DESPUÉS del OPTIMIZE** → evaluar si el solapamiento se resolvió

La progresión de las tres rondas es intencionada:

| Ronda | Rango insertado | Tipo de solapamiento |
|---|---|---|
| 1 | 500K – 1.5M | Localizado, inicio de la tabla |
| 2 | 2.5M – 3.5M | Localizado, zona central |
| 3 | 1M – 4M | Amplio, afecta a múltiples archivos a la vez |

### Helper: función para inspeccionar archivos activos

Definimos una función reutilizable que reconstruye el snapshot vigente de la tabla
leyendo el `_delta_log`.

La lógica es simple: un archivo está **activo** si tiene una entrada `ADD` pero no tiene
una entrada `REMOVE` asociada. Eso es exactamente lo que Delta considera la versión
actual de la tabla en cada momento.

Usaremos esta función después de cada append y después de cada OPTIMIZE para
comparar el estado antes y después.

In [0]:
from pyspark.sql.functions import col

def show_active_files(location):
    """Reconstruct active Delta table snapshot from the _delta_log.
    Returns a DataFrame with one row per active Parquet file,
    showing the min/max id range and size in bytes.
    """
    all_log = spark.read.json(f"{location}/_delta_log/*.json")

    # All files ever written (ADD entries)
    added = (
        all_log.filter(col("add").isNotNull())
               .selectExpr("add.path as path", "add.size as size_bytes", "add.stats as stats")
    )

    # Only subtract removed files if the REMOVE column exists in the log
    # (it won't exist if no OPTIMIZE has run yet)
    if "remove" in all_log.columns:
        removed = (
            all_log.filter(col("remove").isNotNull())
                   .selectExpr("remove.path as path")
        )
        active = added.join(removed, on="path", how="left_anti")
    else:
        # No REMOVEs in the log yet — every ADD is active
        active = added

    active = (
        active.selectExpr(
            "path as file",
            "get_json_object(stats, '$.minValues.id') as id_min",
            "get_json_object(stats, '$.maxValues.id') as id_max",
            "size_bytes"
        )
        .orderBy(col("id_min").cast("long"))
    )

    print(f"Active files: {active.count()}")
    display(active)

print("Helper function defined.")

Helper function defined.


### Ronda 1 — Inserción en rango 500K–1.5M

Insertamos 1 millón de filas con `id` entre 500.000 y 1.499.999.

Este rango cae dentro del primer archivo generado por el OPTIMIZE anterior
(aproximadamente `id_min=0, id_max=804.098`), y también dentro del segundo.

El append no reescribe nada — simplemente añade un archivo nuevo sin ordenar.
El resultado es que **dos archivos cubren el mismo rango de IDs** simultáneamente.

In [0]:
# Round 1: 2M rows scattered across the FULL table range (0-5M)
# Overlaps with ALL existing clustered files simultaneously
df_r1 = (spark.range(0, 2_000_000)
              .selectExpr(
                  "cast(rand() * 5000000 as long) as id",
                  "cast(cast(rand() * 24 as int) + 2000 as long) as year",
                  "cast(rand() * 100000 as int) as salary"
              ))

(df_r1.write
      .format("delta")
      .mode("append")
      .saveAsTable("clustered_employees"))

print("Round 1 appended: 2M rows across full range 0-5M")

Round 1 appended: 2M rows across full range 0-5M


#### Estado ANTES del OPTIMIZE — Ronda 1

Fíjate en que el nuevo archivo del append aparece con su rango completo (500K–1.499.999),
solapando con los archivos ya clusterizados del paso anterior.

> 💡 Esto no significa que la tabla esté corrupta. Delta sabe exactamente qué archivos
> están activos y cuáles no. Pero si lanzas una query con `WHERE id = 600000`,
> Spark tendrá que leer **varios archivos** en lugar de uno solo.

In [0]:
# State BEFORE OPTIMIZE — should show overlap around the 500K-1.5M range
show_active_files(clustered_location)

Active files: 10


file,id_min,id_max,size_bytes
part-00007-a4f915f2-23ba-42a4-b4f3-9c1552e0fe80.c000.snappy.parquet,0,635852,4358560
part-00001-79beccd2-dddd-468d-b974-7aec4d784e7d.c000.snappy.parquet,2,2577331,8315385
part-00003-7c750793-2920-4cb4-b655-a9784a4427c9.c000.snappy.parquet,635853,1245994,4198099
part-00006-3444d6d6-a224-4960-b87e-2b9f783ef94d.c000.snappy.parquet,1245995,1909599,4531366
part-00002-6ac766ff-148c-4230-9e72-d0eb33c1e326.c000.snappy.parquet,1909600,2577334,4557484
part-00001-676e0416-e087-4e01-85f0-06d2c95f192a.c000.snappy.parquet,2577335,3241136,4532492
part-00000-01a6be19-a434-4e63-8ab9-abfa56ec81f4.c000.snappy.parquet,2577336,4999999,7828991
part-00000-1c65658b-f728-484d-99de-fcf35cda1d93.c000.snappy.parquet,3241137,3850740,4194793
part-00005-f2f7186e-ff3f-4f41-843d-801e543d000d.c000.snappy.parquet,3850741,4394614,3784812
part-00004-9e48bb4b-22a1-40bb-a56c-3980096141a4.c000.snappy.parquet,4394615,4999999,4168556


In [0]:
%sql
-- Incremental OPTIMIZE: only rewrites files that need re-clustering
OPTIMIZE clustered_employees;

path,metrics
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 10, 0, false, 0, 0, 1776096291212, 1776096296161, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, List(50470538, false, false, false, 0.8745919689222286, List(0.8745919689222286), 0.6801227678611232, null, 0, 2, 16144376, 16144376, 0, 0, 0, null, log, 4194304, 4194304, 1, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(115, 6, 1926, 0, 0, 0), 2, 1, 5, default, false, 0, null), null)"
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 10, 2, true, 0, 0, 1776096296316, 1776096299094, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, List(50470538, false, false, false, 0.8745919689222286, List(0.8745919689222286), 0.6801227678611232, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 2097152, 4194304, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 1362, 0, 0, 0), 15, 1, 1, null, false, 0, null), null)"


#### Estado DESPUÉS del OPTIMIZE — Ronda 1

El OPTIMIZE incremental ha detectado los archivos afectados y los ha reescrito.

Observa que los archivos de las zonas **no afectadas** (los extremos de la tabla)
**no han cambiado** — tienen el mismo nombre que antes. Solo se han reescrito
los que estaban en el rango solapado.

> 💡 Esto es la clave del OPTIMIZE incremental: coste proporcional al tamaño
> del solapamiento, no al tamaño total de la tabla.

In [0]:
# State AFTER OPTIMIZE — observe how overlap is resolved (or not)
show_active_files(clustered_location)

Active files: 10


file,id_min,id_max,size_bytes
part-00007-a4f915f2-23ba-42a4-b4f3-9c1552e0fe80.c000.snappy.parquet,0,635852,4358560
part-00001-79beccd2-dddd-468d-b974-7aec4d784e7d.c000.snappy.parquet,2,2577331,8315385
part-00003-7c750793-2920-4cb4-b655-a9784a4427c9.c000.snappy.parquet,635853,1245994,4198099
part-00006-3444d6d6-a224-4960-b87e-2b9f783ef94d.c000.snappy.parquet,1245995,1909599,4531366
part-00002-6ac766ff-148c-4230-9e72-d0eb33c1e326.c000.snappy.parquet,1909600,2577334,4557484
part-00001-676e0416-e087-4e01-85f0-06d2c95f192a.c000.snappy.parquet,2577335,3241136,4532492
part-00000-01a6be19-a434-4e63-8ab9-abfa56ec81f4.c000.snappy.parquet,2577336,4999999,7828991
part-00000-1c65658b-f728-484d-99de-fcf35cda1d93.c000.snappy.parquet,3241137,3850740,4194793
part-00005-f2f7186e-ff3f-4f41-843d-801e543d000d.c000.snappy.parquet,3850741,4394614,3784812
part-00004-9e48bb4b-22a1-40bb-a56c-3980096141a4.c000.snappy.parquet,4394615,4999999,4168556


### Ronda 2 — Inserción en rango 2.5M–3.5M

Insertamos 1 millón de filas con `id` entre 2.500.000 y 3.499.999.

Este rango cae en la zona central de la tabla, afectando a archivos
diferentes a los de la Ronda 1.

El comportamiento esperado es el mismo: el append crea un archivo nuevo sin ordenar,
y el OPTIMIZE solo reescribirá los archivos de esa zona central,
dejando intactos los de los extremos y los ya resueltos en la Ronda 1.

In [0]:
# Round 2: another 2M rows scattered across the FULL table range
# Cumulative disorder forces deeper re-clustering
df_r2 = (spark.range(0, 2_000_000)
              .selectExpr(
                  "cast(rand() * 5000000 as long) as id",
                  "cast(cast(rand() * 24 as int) + 2000 as long) as year",
                  "cast(rand() * 100000 as int) as salary"
              ))

(df_r2.write
      .format("delta")
      .mode("append")
      .saveAsTable("clustered_employees"))

print("Round 2 appended: 2M rows across full range 0-5M")

Round 2 appended: 2M rows across full range 0-5M


#### Estado ANTES del OPTIMIZE — Ronda 2

El nuevo archivo del append cubre el rango 2.5M–3.499.999 y solapa
con los archivos centrales de la tabla.

Los archivos de los extremos y los ya consolidados en la Ronda 1
**permanecen intactos** — no han sido afectados por este append.

In [0]:
# State BEFORE OPTIMIZE — new overlap in the 2.5M-3.5M range
show_active_files(clustered_location)

Active files: 14


file,id_min,id_max,size_bytes
part-00007-a4f915f2-23ba-42a4-b4f3-9c1552e0fe80.c000.snappy.parquet,0,635852,4358560
part-00001-79beccd2-dddd-468d-b974-7aec4d784e7d.c000.snappy.parquet,2,2577331,8315385
part-00003-70d700ff-78cc-423e-a73a-35268f610ce2.c000.snappy.parquet,3,1245993,4200046
part-00003-7c750793-2920-4cb4-b655-a9784a4427c9.c000.snappy.parquet,635853,1245994,4198099
part-00002-93ebc46b-8770-464f-9fa0-08cf7baeeb97.c000.snappy.parquet,1245995,2577332,4467599
part-00006-3444d6d6-a224-4960-b87e-2b9f783ef94d.c000.snappy.parquet,1245995,1909599,4531366
part-00002-6ac766ff-148c-4230-9e72-d0eb33c1e326.c000.snappy.parquet,1909600,2577334,4557484
part-00001-676e0416-e087-4e01-85f0-06d2c95f192a.c000.snappy.parquet,2577335,3241136,4532492
part-00001-3ffc4efa-cf6a-4850-9da1-03b598cbd580.c000.snappy.parquet,2577335,3850740,4291310
part-00000-01a6be19-a434-4e63-8ab9-abfa56ec81f4.c000.snappy.parquet,2577336,4999999,7828991


In [0]:
%sql
OPTIMIZE clustered_employees;

path,metrics
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 14, 0, false, 0, 0, 1776096358011, 1776096362036, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, List(67337338, false, false, false, 0.8745919689222286, List(0.8745919689222286), 0.5097641667985152, null, 0, 6, 33011176, 33011176, 0, 0, 0, null, log, 4194304, 4194304, 1, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(122, 7, 1615, 0, 0, 0), 2, 1, 5, default, false, 0, null), null)"
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 14, 6, true, 0, 0, 1776096362210, 1776096365145, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, List(67337338, false, false, false, 0.8745919689222286, List(0.8745919689222286), 0.5097641667985152, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 2097152, 4194304, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 1680, 0, 0, 0), 15, 1, 1, null, false, 0, null), null)"


#### Estado DESPUÉS del OPTIMIZE — Ronda 2

El OPTIMIZE ha consolidado la zona central.

Compara el nombre de los archivos con el estado anterior al OPTIMIZE:
solo los archivos de la zona 2.5M–3.5M tienen nombres nuevos.
El resto no ha cambiado.

> 💡 Cada OPTIMIZE genera nuevos archivos Parquet con nombres únicos.
> Los archivos viejos que fueron reescritos siguen en S3 para soportar **Time Travel**,
> pero Delta ya no los cuenta como activos.

In [0]:
# State AFTER OPTIMIZE — incremental re-clustering of the affected zone
show_active_files(clustered_location)

Active files: 14


file,id_min,id_max,size_bytes
part-00007-a4f915f2-23ba-42a4-b4f3-9c1552e0fe80.c000.snappy.parquet,0,635852,4358560
part-00001-79beccd2-dddd-468d-b974-7aec4d784e7d.c000.snappy.parquet,2,2577331,8315385
part-00003-70d700ff-78cc-423e-a73a-35268f610ce2.c000.snappy.parquet,3,1245993,4200046
part-00003-7c750793-2920-4cb4-b655-a9784a4427c9.c000.snappy.parquet,635853,1245994,4198099
part-00002-93ebc46b-8770-464f-9fa0-08cf7baeeb97.c000.snappy.parquet,1245995,2577332,4467599
part-00006-3444d6d6-a224-4960-b87e-2b9f783ef94d.c000.snappy.parquet,1245995,1909599,4531366
part-00002-6ac766ff-148c-4230-9e72-d0eb33c1e326.c000.snappy.parquet,1909600,2577334,4557484
part-00001-676e0416-e087-4e01-85f0-06d2c95f192a.c000.snappy.parquet,2577335,3241136,4532492
part-00001-3ffc4efa-cf6a-4850-9da1-03b598cbd580.c000.snappy.parquet,2577335,3850740,4291310
part-00000-01a6be19-a434-4e63-8ab9-abfa56ec81f4.c000.snappy.parquet,2577336,4999999,7828991


### Ronda 3 — Inserción amplia en rango 1M–4M

Insertamos 3 millones de filas con `id` entre 1.000.000 y 3.999.999.

Este es el caso más extremo: el rango es tan amplio que solapa simultáneamente
con **múltiples archivos** ya clusterizados en toda la zona central.

Esta situación simula un backfill o una corrección masiva de datos históricos,
que es uno de los escenarios más comunes donde el OPTIMIZE incremental
puede resultar insuficiente.

In [0]:
# Round 3: 3M rows scattered across the FULL table range
# At this point the clustering score should be severely degraded
# OPTIMIZE FULL should be clearly justified after this round
df_r3 = (spark.range(0, 3_000_000)
              .selectExpr(
                  "cast(rand() * 5000000 as long) as id",
                  "cast(cast(rand() * 24 as int) + 2000 as long) as year",
                  "cast(rand() * 100000 as int) as salary"
              ))

(df_r3.write
      .format("delta")
      .mode("append")
      .saveAsTable("clustered_employees"))

print("Round 3 appended: 3M rows across full range 0-5M")

Round 3 appended: 3M rows across full range 0-5M


#### Estado ANTES del OPTIMIZE — Ronda 3

El archivo del append cubre 1M–3.999.999 y solapa con prácticamente
todos los archivos centrales de la tabla a la vez.

Este es el peor escenario para el OPTIMIZE incremental:
tiene que tomar una decisión sobre cuántos archivos merece la pena reescribir
en una sola operación.

In [0]:
# State BEFORE OPTIMIZE — wide overlap across the entire central range
show_active_files(clustered_location)

Active files: 21


file,id_min,id_max,size_bytes
part-00007-a4f915f2-23ba-42a4-b4f3-9c1552e0fe80.c000.snappy.parquet,0,635852,4358560
part-00001-79beccd2-dddd-468d-b974-7aec4d784e7d.c000.snappy.parquet,2,2577331,8315385
part-00001-2dcb33b5-145e-4f2e-9a9f-5c88b83ae46c.c000.snappy.parquet,3,635850,3282436
part-00003-70d700ff-78cc-423e-a73a-35268f610ce2.c000.snappy.parquet,3,1245993,4200046
part-00003-7c750793-2920-4cb4-b655-a9784a4427c9.c000.snappy.parquet,635853,1245994,4198099
part-00002-58a2b7bb-54a4-4c16-9b7c-79c98ed69d37.c000.snappy.parquet,635853,1245993,3158981
part-00006-3444d6d6-a224-4960-b87e-2b9f783ef94d.c000.snappy.parquet,1245995,1909599,4531366
part-00005-722c20ba-65ce-4d47-8d3b-048ce63531bc.c000.snappy.parquet,1245995,1909597,3408466
part-00002-93ebc46b-8770-464f-9fa0-08cf7baeeb97.c000.snappy.parquet,1245995,2577332,4467599
part-00004-1b4c708c-924c-432c-a620-c6aee900538c.c000.snappy.parquet,1909600,2577331,3414356


In [0]:
%sql
OPTIMIZE clustered_employees;

path,metrics
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 21, 0, false, 0, 0, 1776321660511, 1776321672049, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, List(92783615, false, false, false, 0.8745919689222286, List(0.8745919689222286), 0.3699593080092859, null, 0, 7, 39219253, 39219253, 6, 19238200, 19238200, null, log, 4194304, 4194304, 1, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(331, 605, 5366, 409, 0, 0), 2, 1, 5, default, false, 0, null), null)"
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 21, 13, true, 0, 0, 1776321672243, 1776321675769, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, List(92783615, false, false, false, 0.8745919689222286, List(0.8745919689222286), 0.3699593080092859, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 2097152, 4194304, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 1567, 0, 0, 0), 15, 1, 1, null, false, 0, null), null)"


#### Estado DESPUÉS del OPTIMIZE — Ronda 3

Observa el resultado: el OPTIMIZE ha mejorado la situación, pero es posible que
quede **solapamiento residual** en algunos rangos.

Esto es el comportamiento documentado y esperado de Liquid Clustering:

> *El OPTIMIZE incremental es una heurística de coste/beneficio,
> no una garantía de orden perfecto.*

El query engine lo gestiona correctamente igualmente, usando las estadísticas
del `_delta_log` para hacer **file pruning** — descarta archivos cuyos rangos
no intersectan con el filtro de la query, aunque haya solapamiento.

Para eliminar el solapamiento completamente existe `OPTIMIZE FULL`, que veremos a continuación.

In [0]:
# State AFTER OPTIMIZE — observe residual overlap after a wide insertion
show_active_files(clustered_location)

Active files: 21


file,id_min,id_max,size_bytes
part-00007-a4f915f2-23ba-42a4-b4f3-9c1552e0fe80.c000.snappy.parquet,0,635852,4358560
part-00001-79beccd2-dddd-468d-b974-7aec4d784e7d.c000.snappy.parquet,2,2577331,8315385
part-00001-2dcb33b5-145e-4f2e-9a9f-5c88b83ae46c.c000.snappy.parquet,3,635850,3282436
part-00003-70d700ff-78cc-423e-a73a-35268f610ce2.c000.snappy.parquet,3,1245993,4200046
part-00003-7c750793-2920-4cb4-b655-a9784a4427c9.c000.snappy.parquet,635853,1245994,4198099
part-00002-58a2b7bb-54a4-4c16-9b7c-79c98ed69d37.c000.snappy.parquet,635853,1245993,3158981
part-00006-3444d6d6-a224-4960-b87e-2b9f783ef94d.c000.snappy.parquet,1245995,1909599,4531366
part-00005-722c20ba-65ce-4d47-8d3b-048ce63531bc.c000.snappy.parquet,1245995,1909597,3408466
part-00002-93ebc46b-8770-464f-9fa0-08cf7baeeb97.c000.snappy.parquet,1245995,2577332,4467599
part-00004-1b4c708c-924c-432c-a620-c6aee900538c.c000.snappy.parquet,1909600,2577331,3414356


### 6.4 OPTIMIZE FULL — reescritura completa

Cuando el solapamiento acumulado es tan alto que el file pruning ya no es efectivo,
existe la opción de forzar una reescritura completa:

```sql
OPTIMIZE clustered_employees FULL;
```

**¿Qué hace exactamente?**  
Ignora qué archivos ya estaban clusterizados y reescribe **toda la tabla** desde cero,
garantizando rangos completamente no solapados.

**¿Cuándo tiene sentido usarlo?**

| Escenario | ¿OPTIMIZE FULL? |
|---|---|
| Backfill masivo de datos históricos | ✅ Sí |
| Solapamiento generalizado por múltiples inserciones | ✅ Sí |
| Tabla con actualizaciones frecuentes y pequeñas | ❌ No, demasiado costoso |
| Mantenimiento rutinario | ❌ No, el incremental es suficiente |

> ⚠️ El coste de `OPTIMIZE FULL` es equivalente a reescribir toda la tabla desde cero.
> En tablas grandes esto puede tardar mucho tiempo y consumir muchos recursos de compute.

> 💡 **Conclusión final del ejercicio:** Liquid Clustering es una estrategia flexible.
> El solapamiento parcial es normal y el engine lo gestiona bien. `OPTIMIZE FULL` existe
> para casos extremos, no como práctica habitual.

In [0]:
%sql
-- Full rewrite: all files are rewritten and ranges become non-overlapping
-- Use only when incremental OPTIMIZE is no longer effective
OPTIMIZE clustered_employees FULL;

path,metrics
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 21, 0, false, 0, 0, 1776096505882, 1776096510610, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, List(92783615, false, false, true, 0.8745919689222286, List(0.8745919689222286), 0.3699593080092859, null, 0, 7, 39219253, 39219253, 6, 19238200, 19238200, null, log, 4194304, 4194304, 1, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(91, 321, 1751, 307, 0, 0), 2, 1, 5, default, false, 0, null), null)"
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 21, 13, true, 0, 0, 1776096510773, 1776096513493, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, List(92783615, false, false, false, 0.8745919689222286, List(0.8745919689222286), 0.3699593080092859, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 2097152, 4194304, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 1325, 0, 0, 0), 15, 1, 1, null, false, 0, null), null)"
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 21, 0, true, 0, 0, 1776096513663, 1776096514093, 8, 0, null, List(0, 0), SNAPPY, 3, 3, 0, 0, null, null)"


In [0]:
# Final state after FULL OPTIMIZE: clean non-overlapping ranges
show_active_files(clustered_location)

Active files: 21


file,id_min,id_max,size_bytes
part-00007-a4f915f2-23ba-42a4-b4f3-9c1552e0fe80.c000.snappy.parquet,0,635852,4358560
part-00001-79beccd2-dddd-468d-b974-7aec4d784e7d.c000.snappy.parquet,2,2577331,8315385
part-00001-2dcb33b5-145e-4f2e-9a9f-5c88b83ae46c.c000.snappy.parquet,3,635850,3282436
part-00003-70d700ff-78cc-423e-a73a-35268f610ce2.c000.snappy.parquet,3,1245993,4200046
part-00003-7c750793-2920-4cb4-b655-a9784a4427c9.c000.snappy.parquet,635853,1245994,4198099
part-00002-58a2b7bb-54a4-4c16-9b7c-79c98ed69d37.c000.snappy.parquet,635853,1245993,3158981
part-00006-3444d6d6-a224-4960-b87e-2b9f783ef94d.c000.snappy.parquet,1245995,1909599,4531366
part-00005-722c20ba-65ce-4d47-8d3b-048ce63531bc.c000.snappy.parquet,1245995,1909597,3408466
part-00002-93ebc46b-8770-464f-9fa0-08cf7baeeb97.c000.snappy.parquet,1245995,2577332,4467599
part-00004-1b4c708c-924c-432c-a620-c6aee900538c.c000.snappy.parquet,1909600,2577331,3414356


## Resumen

| Concepto | Partitioning | Liquid Clustering |
|---|---|---|
| Organización física | Subdirectorios por valor de columna | Orden dentro de archivos Parquet |
| Compatible entre sí | ❌ No | — |
| Se activa al escribir | ✅ Sí | ❌ No (requiere OPTIMIZE) |
| Coste de mantenimiento | Alto (muchos ficheros pequeños) | Bajo (OPTIMIZE incremental) |
| Ideal para | Filtros exactos por columna de baja cardinalidad | Filtros por rango en columnas de alta cardinalidad |

> **Conclusión:** Liquid Clustering es la estrategia recomendada por Databricks para tablas nuevas en la mayoría de casos de uso.  
> Partitioning sigue siendo válido para casos muy específicos (p. ej. retención por fecha, compliance) pero introduce complejidad operativa adicional.